### Middleware

Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:

- Tracking agent behavior with logging, analytics, and debugging.
- Transforming prompts, tool selection, and output formatting.
- Adding retries, fallbacks, and early termination logic.
- Applying rate limits, guardrails, and PII detection.

In [1]:
import os 

from dotenv import load_dotenv

load_dotenv()

os.environ['GOOGLE_API_KEY'] = os.getenv('GOOGLE_API_KEY')

### Summarization Middleware

Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:

- Long-running conversations that exceed context windows.
- Multi-turn dialogues with extensive history.
- Applications where preserving full conversation context matters.

In [2]:
from langchain.agents import create_agent

from langchain.agents.middleware import SummarizationMiddleware

from langgraph.checkpoint.memory import InMemorySaver

from langchain.messages import HumanMessage, SystemMessage

## Message based summarization

os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')


agent= create_agent(
    model='gpt-4o-mini',
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model='gpt-4o-mini',
            trigger=("messages", 5),
            keep=("messages", 2)
        )
    ]
)

In [3]:
## run with thread id

config={"configurable" : {"thread_id": "test-1"}}


In [4]:
# Alternative test data
questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?",
]

for q in questions:

    response = agent.invoke({"messages": [HumanMessage(content=q)] }, config)

    print(f"Messages:{response}")

    print(f"Messages: {len(response['messages'])}")


Messages:{'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='67982603-1af3-42f0-91f5-4046455bc847'), AIMessage(content='2 + 2 equals 4.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 14, 'total_tokens': 22, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_0185f8e998', 'id': 'chatcmpl-DtbWrXJ2KIvKX4Wm7zfeqmeCQ03yw', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019ef019-a457-72e2-bde7-aa9909899678-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 14, 'output_tokens': 8, 'total_tokens': 22, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {

In [5]:
### Based on token size

from langchain.agents import create_agent

from langchain.agents.middleware import SummarizationMiddleware

from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to user more tokens"""

    return f"""Top hotels in {city}:
1. The Grand Plaza - A 5-star luxury hotel in the heart of {city}, featuring rooftop dining, a full-service spa, and panoramic skyline views from every suite.
2. Riverside Boutique Inn - A charming mid-range stay along the {city} waterfront, known for its locally-sourced breakfast, cozy reading lounge, and walkable access to museums and cafes.
3. Skyline Business Suites - A modern hotel near the {city} convention district, offering high-speed internet, dedicated workspaces, and 24/7 concierge service tailored for business travelers."""


agent = create_agent(
    model='gpt-4o-mini',
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model='gpt-4o-mini',
            trigger=("tokens", 550),
            keep=("tokens", 200)
        )
    ]
)


## run with thread id

config={"configurable" : {"thread_id": "test-1"}}

# Token counter (approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars //4 # 4 chars = 1 token


In [6]:
## run test

cities = ['paris', 'london', 'Hyderabad']

for city in cities:
    response = agent.invoke(
        {'messages' : [HumanMessage(content=f"find hotels in {city}")]},
        config=config
    )

    tokens = count_tokens(response['messages'])
    print(f"{city}: _ {tokens} {len(response['messages'])} messages")

    print(f"{(response['messages'])}")

paris: _ 326 4 messages
[HumanMessage(content='find hotels in paris', additional_kwargs={}, response_metadata={}, id='4701d40d-396e-4e70-b87d-4ab17acc1381'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 53, 'total_tokens': 69, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_86eb24cbce', 'id': 'chatcmpl-DtbXINxhQqcenfKD5sPX3WDkQaYOK', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019ef01a-0b12-7651-a664-3108eb2c0642-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'paris'}, 'id': 'call_1wz9QObbFl5xzeGq9BP57QdK', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 53, 'outp

In [7]:
### Fraction

### Based on token size

from langchain.agents import create_agent

from langchain.agents.middleware import SummarizationMiddleware

from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to user more tokens"""

    return f"""Top hotels in {city} The Grand Plaza - A 5-star luxury hotel in the heart of {city}, featuring rooftop dining, a full-service spa, and panoramic skyline views from every suite."""


agent = create_agent(
    model='gpt-4o-mini',
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model='gpt-4o-mini',
            trigger=("fraction", 0.005), # 0.5% = ~ 640 tokens
            keep=("fraction", 0.002), # 0.2% ~ 256 tokens
        )
    ]
)


## run with thread id

config={"configurable" : {"thread_id": "test-1"}}

# Token counter (approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars //4 # 4 chars = 1 token


In [8]:
## run test

cities = ['paris', 'london', 'Hyderabad']

for city in cities:
    response = agent.invoke(
        {'messages' : [HumanMessage(content=f"find hotels in {city}")]},
        config=config
    )

    tokens = count_tokens(response['messages'])
    fraction = tokens / 128000 # gpt-40-mini context
    print(f"{city}: _ {tokens} {len(response['messages'])} messages")

    print(f"{(response['messages'])}")

paris: _ 124 4 messages
[HumanMessage(content='find hotels in paris', additional_kwargs={}, response_metadata={}, id='60be84f7-ce10-4d50-bdcb-2b915606bbd5'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 53, 'total_tokens': 68, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_86eb24cbce', 'id': 'chatcmpl-DtbXi7lFlqE6KzsKE763PCgB4uDe5', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019ef01a-7355-7f52-9d76-ca6ef590285e-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Paris'}, 'id': 'call_wavYEc2JbsiqKAZiuYO05SEQ', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 53, 'outp

### Human in the Loop Middleware


In [9]:
from langchain.agents import create_agent

from langchain.agents.middleware import HumanInTheLoopMiddleware

from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id:str) -> str:
    """Mock function to read an email"""

    return f"Email content for ID: {email_id}"


def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email"""

    return f"Email sent to {recipient} with subject '{subject}' and body: {body}"

In [10]:
agent = create_agent(
    model='gpt-4o-mini',
    checkpointer=InMemorySaver(),
    tools=[read_email_tool, send_email_tool],
    middleware=[
       HumanInTheLoopMiddleware(
           interrupt_on={
           "send_email_tool": {
               "allowed_decisions" : ["approve", "edit", "reject"]
           },
            "read_email_tool": False,
          
            }
    
        )
    ]
)

In [11]:
config={"configurable" : {"thread_id": "test-1"}}

#Step1: Request

result = agent.invoke(
    
    {"messages" : HumanMessage(content="Send message john@test.com with subject hello and body hi")},
    config=config
    )



result

{'messages': [HumanMessage(content='Send message john@test.com with subject hello and body hi', additional_kwargs={}, response_metadata={}, id='c9a1f60b-dc19-48fe-8aaa-aef6d74ec256'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 87, 'total_tokens': 112, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_997b5b4ee9', 'id': 'chatcmpl-DtbXsnZoNZhjBkRxmiZuViFEMLgMD', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019ef01a-9be1-7b33-a1fb-efd5670bf9c6-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recipient': 'john@test.com', 'subject': 'hello', 'body': 'hi'}, 'id': 'call_JpOmJ86QpLJ3fE08h5EeSLvd', 'type': '

In [12]:
#Step 2: Approve

from langgraph.types import Command

if "__interrupt__" in result:
    print("Paused Approving.....")

    result = agent.invoke(
        Command(
            resume= {
                "decisions" : [
                    {"type" : "approve"}
                ]
            }
        ),
        config=config
    )

    print(result['messages'][-1].content)

Paused Approving.....
The email has been successfully sent to john@test.com with the subject "hello" and the body "hi".


# Reject


In [13]:


config={"configurable" : {"thread_id": "test-reject"}}

#Step1: Request

result = agent.invoke(
    
    {"messages" : HumanMessage(content="Send message john@test.com with subject hello and body hi")},
    config=config
    )



#Step 2: Reject

from langgraph.types import Command

if "__interrupt__" in result:
    print("Paused Approving.....")

    result = agent.invoke(
        Command(
            resume= {
                "decisions" : [
                    {"type" : "reject"}
                ]
            }
        ),
        config=config
    )

    print(result['messages'][-1].content)

Paused Approving.....
It seems there was an issue sending the email. Would you like to try again or is there something else you would like to do?


# Edit

In [14]:
import uuid
config = {"configurable": {"thread_id": f"test-edit-{uuid.uuid4()}"}}


#Step1: Request

result = agent.invoke(
    
    {"messages" : HumanMessage(content="Send message wrong@test.com with subject hello and body hi")},
    config=config
    )


print(result) 

#Step 2: Reject

from langgraph.types import Command

if "__interrupt__" in result:
    print("Paused Approving.....")

    result = agent.invoke(
       Command(resume={"decisions": [{
    "type": "edit",
    "edited_action": {
        "name": "send_email_tool",
        "args": {"recipient": "ksk@gmail.com", "subject": "Hi Ksk", "body": "How are you"}
    }
}]}),
        config=config
    )

    print(result['messages'][-1].content)

{'messages': [HumanMessage(content='Send message wrong@test.com with subject hello and body hi', additional_kwargs={}, response_metadata={}, id='c2fed801-52b6-4628-86e5-f6f4a4cec0a2'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 87, 'total_tokens': 112, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_997b5b4ee9', 'id': 'chatcmpl-DtbXyq5lZneiry2qqPKTCW8BwCAYS', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019ef01a-b30c-7341-a72d-f53a5aa51584-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recipient': 'wrong@test.com', 'subject': 'hello', 'body': 'hi'}, 'id': 'call_v6ev9ypnIQqT93W4jzk05tif', 'type': '